# sklearn Integration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/forge-features/forge/blob/main/notebooks/04_sklearn_integration.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/forge-features/forge/main?labpath=notebooks/04_sklearn_integration.ipynb)

This notebook shows how Forge integrates seamlessly with scikit-learn.

## What you'll learn

1. Using Forge in sklearn Pipelines
2. Cross-validation with Forge
3. Hyperparameter tuning with GridSearchCV
4. Saving and loading pipelines

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

np.random.seed(42)

# Create sample data
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    random_state=42
)

# Add some categorical features
X = pd.DataFrame(X, columns=[f'num_{i}' for i in range(10)])
X['category_a'] = np.random.choice(['A', 'B', 'C'], len(X))
X['category_b'] = np.random.choice(['X', 'Y'], len(X))
y = pd.Series(y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## Using Forge in sklearn Pipelines

Forge transformers work directly with sklearn's Pipeline:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from forge import AutoFeatureTransformer

# Create pipeline with Forge and a classifier
pipeline = Pipeline([
    ('features', AutoFeatureTransformer(max_features=50)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Fit the pipeline
pipeline.fit(X_train, y_train)

# Make predictions
y_pred = pipeline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\nFeatures generated: {len(pipeline['features'].get_feature_names_out())}")

## Cross-Validation

Use cross-validation to evaluate the pipeline:

In [ ]:
from sklearn.model_selection import cross_val_score

# Create a fresh pipeline for cross-validation
cv_pipeline = Pipeline([
    ('features', AutoFeatureTransformer(max_features=30)),
    ('classifier', RandomForestClassifier(n_estimators=50, random_state=42))
])

# 5-fold cross-validation
scores = cross_val_score(cv_pipeline, X, y, cv=5, scoring='accuracy')

print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

## Hyperparameter Tuning with GridSearchCV

Tune both Forge and model hyperparameters:

In [ ]:
from sklearn.model_selection import GridSearchCV

# Create pipeline
search_pipeline = Pipeline([
    ('features', AutoFeatureTransformer()),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Define parameter grid
param_grid = {
    'features__max_features': [20, 50, 100],
    'features__selection_method': ['importance', 'mutual_info'],
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10, None]
}

# Grid search
grid_search = GridSearchCV(
    search_pipeline,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")
print(f"Test score: {grid_search.score(X_test, y_test):.4f}")

## Comparing Different Classifiers

Test multiple classifiers with the same feature engineering:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Define classifiers to test
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

results = {}

for name, clf in classifiers.items():
    pipeline = Pipeline([
        ('features', AutoFeatureTransformer(max_features=30)),
        ('classifier', clf)
    ])
    
    scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
    results[name] = scores.mean()
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

print(f"\nBest classifier: {max(results, key=results.get)}")

## Saving and Loading Pipelines

Save trained pipelines for later use:

In [ ]:
import joblib

# Train a pipeline
final_pipeline = Pipeline([
    ('features', AutoFeatureTransformer(max_features=50)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
final_pipeline.fit(X_train, y_train)

# Save to file
joblib.dump(final_pipeline, 'forge_pipeline.joblib')
print("Pipeline saved to 'forge_pipeline.joblib'")

# Load and use
loaded_pipeline = joblib.load('forge_pipeline.joblib')
loaded_pred = loaded_pipeline.predict(X_test)
print(f"Loaded pipeline accuracy: {accuracy_score(y_test, loaded_pred):.4f}")

In [ ]:
# Clean up
import os
if os.path.exists('forge_pipeline.joblib'):
    os.remove('forge_pipeline.joblib')
    print("Cleaned up saved pipeline file")

## Next Steps

- [05_kaggle_workflow.ipynb](05_kaggle_workflow.ipynb) - Complete Kaggle competition workflow